# Harfi: Llama 3.2 1B v5, part 1 of 2 (Kaggle)

v4's recipe (**Llama 3.2 1B**, LoRA rank 32, alpha 64, lr 2e-4) with 2 epochs instead of 3 (v4's best was epoch 2), on
the `harfi-v6` export (144,986 examples, spelling v2.0 + round 6), the data Qwen v2 trained on. About 4.8 hours on a T4,
so it trains in two parts: **part 1 on Kaggle** (`finetune_llama_v5.ipynb`, stops itself after about 2.5 hours with a
checkpoint) and **part 2 on Colab** (`finetune_llama_v5_colab.ipynb`, goes on from that checkpoint and makes the GGUF
files). A run stopped and resumed ends with the same weights as one uninterrupted run (checked on the laptop with a
tiny model).

Licence: Llama 3.2 Community License (Unsloth's ungated copy of Meta's weights; accepted 14 Oct 2026).

**How to run:**
1. Add Input -> the `harfi-v6` dataset (the same one as Qwen v2), and only it.
2. Settings -> Accelerator -> **GPU T4** · **Internet on**.
3. **Save Version -> Save & Run All (Commit)**. Training stops by itself after `BUDGET` minutes (145 by default, for
   about 2h50 of quota left; lower it if you have less: setup and the last validation add about 10 minutes).
4. From the version's **Output** tab, download **`llama-v5-part1.zip`** (a few hundred MB) and put it in Google Drive,
   in a folder `harfi`, next to `arabizi-train-v6.zip`. Then run part 2 on Colab.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU: the same effective batch (32) as v4


In [ ]:
REF = "data/round6-qwen-v2"  # the branch with checkpoints and resuming (main once it is merged)
# Fresh copy of the code on every run (an old clone would otherwise be kept and run).
!rm -rf /kaggle/working/repo
!git clone --quiet --depth 1 --branch "$REF" https://github.com/Mustafa-Abbara/arabizi-keyboard /kaggle/working/repo
!git -C /kaggle/working/repo log -1 --format='repo commit: %h %s'
!pip install --quiet -r /kaggle/working/repo/train/requirements.txt


In [ ]:
import glob
import json
import subprocess
import sys

exports = glob.glob("/kaggle/input/**/export.json", recursive=True)
assert len(exports) == 1, f"add exactly one dataset with export.json (found {exports})"
DATA = os.path.dirname(exports[0])
with open(exports[0]) as f:
    summary = json.load(f)
print(DATA, "|", summary["rows_total"], "examples | AI share", summary["ai_share"], "| prompt v", summary["prompt_version"])
assert summary["rows_total"] == 144986, "this should be the harfi-v6 export (spelling v2.0 + round 6)"
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"


In [ ]:
# v4's settings, 2 epochs, a checkpoint every 538 steps (1/8 of an epoch, about 17 min) and a time budget: training
# stops with a checkpoint after BUDGET minutes, so the session ends normally and its output is kept.
BUDGET = 145
BASE, TAG = "llama3.2-1b", "v5"

print(f"=== {BASE} {TAG}, part 1", flush=True)
subprocess.run([sys.executable, f"{REPO}/train/finetune_lora.py", "--base", BASE, "--data", DATA, "--out", OUT,
                "--tag", TAG, "--rank", "32", "--alpha", "64", "--epochs", "2",
                "--save-steps", "538", "--time-budget", str(BUDGET)], check=True)
run = max(glob.glob(f"{OUT}/*-{BASE}-lora-{TAG}"))
with open(f"{run}/PARTIAL.json") as f:
    print(json.load(f))


## Download for part 2

One zip with the run folder (its last checkpoint and `PARTIAL.json`). The cloned code is removed so the Output holds only this.


In [ ]:
!rm -rf /kaggle/working/repo
import shutil
import zipfile

shutil.make_archive("/kaggle/working/llama-v5-part1", "zip", OUT)
with zipfile.ZipFile("/kaggle/working/llama-v5-part1.zip") as z:
    print(os.path.getsize("/kaggle/working/llama-v5-part1.zip") // 2**20, "MB:", *sorted({n.split("/checkpoint-")[0] + ("/checkpoint-" + n.split("/checkpoint-")[1].split("/")[0] if "/checkpoint-" in n else "") for n in z.namelist()}), sep="
  ")
